# Stage 3 & 4 — Baseline and Advanced Model Training

**Stage 3:** Baseline models (Logistic Regression, Decision Tree, Naive Bayes) — establish a performance floor  
**Stage 4:** Advanced models (Random Forest, XGBoost, LightGBM) + hyperparameter tuning

All models are evaluated via **5-fold stratified cross-validation** on the training set.  
The validation set is used for final head-to-head comparison.  
The test set is **not touched** — reserved for `evaluation.ipynb`.

**Primary metric:** F1-macro (balances precision and recall across both classes)  
**Secondary:** ROC-AUC, per-class precision/recall

## 1. Imports & Setup

In [ ]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.metrics import (
    classification_report, roc_auc_score,
    ConfusionMatrixDisplay, f1_score
)
from scipy.stats import randint, uniform

import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
CV_FOLDS     = 5
print('Libraries loaded.')
print(f'XGBoost {xgb.__version__} | LightGBM {lgb.__version__}')

## 2. Load Processed Data

In [ ]:
# Scaled splits  → for linear models (LR, NB)
train_s = pd.read_parquet('data/processed/train.parquet')
val_s   = pd.read_parquet('data/processed/val.parquet')

# Unscaled splits → for tree-based models (DT, RF, XGBoost, LightGBM)
train_u = pd.read_parquet('data/processed/train_unscaled.parquet')
val_u   = pd.read_parquet('data/processed/val_unscaled.parquet')

FEATURE_COLS = joblib.load('data/processed/feature_cols.joblib')

X_train_s, y_train = train_s[FEATURE_COLS].values, train_s['label'].astype(int).values
X_val_s,   y_val   = val_s[FEATURE_COLS].values,   val_s['label'].astype(int).values
X_train_u           = train_u[FEATURE_COLS].values
X_val_u             = val_u[FEATURE_COLS].values

# Class imbalance ratio (used for XGBoost scale_pos_weight)
spam_count  = (y_train == 0).sum()
legit_count = (y_train == 1).sum()
scale_pos_weight = spam_count / legit_count   # >1 → upweight the positive (legit) class

print(f'Train: {len(y_train):,}  (spam={spam_count}, legit={legit_count})')
print(f'Val:   {len(y_val):,}    (spam={(y_val==0).sum()}, legit={(y_val==1).sum()})')
print(f'Features: {len(FEATURE_COLS)} → {FEATURE_COLS}')
print(f'scale_pos_weight: {scale_pos_weight:.3f}')

## 3. Helper: Cross-Validation Evaluator

In [ ]:
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

def cv_evaluate(name, model, X, y):
    """Run stratified k-fold CV and return a summary dict."""
    scores = cross_validate(
        model, X, y, cv=cv,
        scoring=['f1_macro', 'roc_auc', 'f1_weighted'],
        return_train_score=False, n_jobs=-1
    )
    result = {
        'Model':        name,
        'CV F1-macro':  scores['test_f1_macro'].mean(),
        'CV F1-macro±': scores['test_f1_macro'].std(),
        'CV ROC-AUC':   scores['test_roc_auc'].mean(),
        'CV ROC-AUC±':  scores['test_roc_auc'].std(),
    }
    print(f"  {name:35s}  F1-macro={result['CV F1-macro']:.4f} ±{result['CV F1-macro±']:.4f}  "
          f"AUC={result['CV ROC-AUC']:.4f} ±{result['CV ROC-AUC±']:.4f}")
    return result

results = []  # collect all model results here
print('CV evaluator ready.')

---
## Stage 3 — Baseline Models

Three simple models trained on the full training set with 5-fold CV.  
Purpose: establish a lower bound that advanced models must beat.

### 3.1 Logistic Regression

In [ ]:
lr = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=RANDOM_STATE
)
results.append(cv_evaluate('Logistic Regression', lr, X_train_s, y_train))

### 3.2 Decision Tree

In [ ]:
dt = DecisionTreeClassifier(
    max_depth=6,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=RANDOM_STATE
)
results.append(cv_evaluate('Decision Tree', dt, X_train_u, y_train))

### 3.3 Gaussian Naive Bayes

In [ ]:
nb = GaussianNB()
results.append(cv_evaluate('Gaussian Naive Bayes', nb, X_train_s, y_train))

### 3.4 Baseline Summary

In [ ]:
baseline_df = pd.DataFrame(results)
print('=== Baseline CV Results ===')
print(baseline_df[['Model', 'CV F1-macro', 'CV F1-macro±', 'CV ROC-AUC']].to_string(index=False))

---
## Stage 4 — Advanced Models

Tree ensembles and gradient boosting. These use the **unscaled** splits — tree-based models are scale-invariant.

### 4.1 Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=3,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
results.append(cv_evaluate('Random Forest', rf, X_train_u, y_train))

### 4.2 XGBoost

In [ ]:
xgb_clf = xgb.XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0
)
results.append(cv_evaluate('XGBoost', xgb_clf, X_train_u, y_train))

### 4.3 LightGBM

In [ ]:
lgb_clf = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)
results.append(cv_evaluate('LightGBM', lgb_clf, X_train_u, y_train))

### 4.4 All-Model CV Comparison

In [ ]:
results_df = pd.DataFrame(results).sort_values('CV F1-macro', ascending=False).reset_index(drop=True)
print('=== All Models — CV F1-macro (descending) ===')
print(results_df[['Model', 'CV F1-macro', 'CV F1-macro±', 'CV ROC-AUC', 'CV ROC-AUC±']].to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(results_df))]
bars = ax.barh(results_df['Model'], results_df['CV F1-macro'], xerr=results_df['CV F1-macro±'],
               color=colors, edgecolor='white', capsize=4)
ax.set_xlabel('CV F1-macro (mean ± std)')
ax.set_title('Model Comparison — 5-Fold Stratified CV')
ax.set_xlim(0, 1.05)
for bar, val in zip(bars, results_df['CV F1-macro']):
    ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## Stage 4 (cont.) — Hyperparameter Tuning

Tune the **top-performing model** from CV. We use `RandomizedSearchCV` (50 iterations, 5-fold) optimising for F1-macro.

In [ ]:
best_model_name = results_df.iloc[0]['Model']
print(f'Best CV model: {best_model_name} — tuning this one.')

### 4.5 Tune XGBoost

In [ ]:
xgb_param_dist = {
    'n_estimators':      randint(100, 600),
    'max_depth':         randint(3, 10),
    'learning_rate':     uniform(0.01, 0.2),
    'subsample':         uniform(0.6, 0.4),
    'colsample_bytree':  uniform(0.5, 0.5),
    'min_child_weight':  randint(1, 10),
    'gamma':             uniform(0, 0.5),
    'reg_alpha':         uniform(0, 1),
    'reg_lambda':        uniform(0.5, 2),
}

xgb_base = xgb.XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0
)

xgb_search = RandomizedSearchCV(
    xgb_base, xgb_param_dist,
    n_iter=50, cv=cv,
    scoring='f1_macro',
    random_state=RANDOM_STATE,
    n_jobs=-1, verbose=1
)
xgb_search.fit(X_train_u, y_train)

print(f'\nBest CV F1-macro (XGBoost tuned): {xgb_search.best_score_:.4f}')
print('Best params:')
for k, v in xgb_search.best_params_.items():
    print(f'  {k}: {v}')

### 4.6 Tune LightGBM

In [ ]:
lgb_param_dist = {
    'n_estimators':     randint(100, 600),
    'max_depth':        randint(3, 10),
    'learning_rate':    uniform(0.01, 0.2),
    'subsample':        uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.5, 0.5),
    'min_child_samples':randint(5, 50),
    'reg_alpha':        uniform(0, 1),
    'reg_lambda':       uniform(0, 2),
    'num_leaves':       randint(20, 100),
}

lgb_base = lgb.LGBMClassifier(
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)

lgb_search = RandomizedSearchCV(
    lgb_base, lgb_param_dist,
    n_iter=50, cv=cv,
    scoring='f1_macro',
    random_state=RANDOM_STATE,
    n_jobs=-1, verbose=1
)
lgb_search.fit(X_train_u, y_train)

print(f'\nBest CV F1-macro (LightGBM tuned): {lgb_search.best_score_:.4f}')
print('Best params:')
for k, v in lgb_search.best_params_.items():
    print(f'  {k}: {v}')

---
## 5. Validation Set Evaluation

All models (baselines + tuned advanced) evaluated on the held-out **validation set** to pick the final model.  
The test set is still untouched.

In [ ]:
def val_evaluate(name, model, X_train, y_train, X_val, y_val):
    """Fit on full train set, evaluate on val set."""
    model.fit(X_train, y_train)
    y_pred  = model.predict(X_val)
    y_proba = model.predict_proba(X_val)[:, 1]
    return {
        'Model':          name,
        'Val F1-macro':   f1_score(y_val, y_pred, average='macro'),
        'Val ROC-AUC':    roc_auc_score(y_val, y_proba),
        'Val F1-spam':    f1_score(y_val, y_pred, pos_label=0),
        'Val F1-legit':   f1_score(y_val, y_pred, pos_label=1),
        '_model':         model,
        '_X_train':       X_train,
    }

val_results = [
    val_evaluate('Logistic Regression',  lr,                   X_train_s, y_train, X_val_s, y_val),
    val_evaluate('Decision Tree',        dt,                   X_train_u, y_train, X_val_u, y_val),
    val_evaluate('Gaussian Naive Bayes', nb,                   X_train_s, y_train, X_val_s, y_val),
    val_evaluate('Random Forest',        rf,                   X_train_u, y_train, X_val_u, y_val),
    val_evaluate('XGBoost (tuned)',      xgb_search.best_estimator_, X_train_u, y_train, X_val_u, y_val),
    val_evaluate('LightGBM (tuned)',     lgb_search.best_estimator_, X_train_u, y_train, X_val_u, y_val),
]

val_df = pd.DataFrame(val_results).drop(columns=['_model', '_X_train'])
val_df = val_df.sort_values('Val F1-macro', ascending=False).reset_index(drop=True)
print('=== Validation Set Results ===')
print(val_df.to_string(index=False))

### 5.1 Validation Comparison Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
metrics = ['Val F1-macro', 'Val ROC-AUC']
titles  = ['Validation F1-macro', 'Validation ROC-AUC']

for ax, metric, title in zip(axes, metrics, titles):
    sorted_df = val_df.sort_values(metric)
    colors = ['#2ecc71' if i == len(sorted_df) - 1 else '#3498db'
              for i in range(len(sorted_df))]
    bars = ax.barh(sorted_df['Model'], sorted_df[metric], color=colors, edgecolor='white')
    ax.set_xlabel(metric)
    ax.set_title(title)
    ax.set_xlim(0, 1.05)
    for bar, val in zip(bars, sorted_df[metric]):
        ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', fontsize=8)

plt.suptitle('All Models — Validation Set Performance', fontsize=13)
plt.tight_layout()
plt.show()

### 5.2 Confusion Matrices — Top 3 Models

In [ ]:
top3_names = val_df['Model'].head(3).tolist()
top3_data  = {r['Model']: r for r in val_results}

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, name in zip(axes, top3_names):
    entry  = top3_data[name]
    model  = entry['_model']
    X_tr   = entry['_X_train']
    # model already fitted from val_evaluate; use appropriate val set
    X_v    = X_val_u if name not in ('Logistic Regression', 'Gaussian Naive Bayes') else X_val_s
    ConfusionMatrixDisplay.from_predictions(
        y_val, model.predict(X_v),
        display_labels=['Spam (0)', 'Legit (1)'],
        cmap='Blues', ax=ax
    )
    ax.set_title(name)

plt.suptitle('Confusion Matrices — Top 3 Models (Validation Set)', fontsize=12)
plt.tight_layout()
plt.show()

### 5.3 Classification Report — Best Model

In [ ]:
best_name  = val_df.iloc[0]['Model']
best_entry = top3_data[best_name]
best_model = best_entry['_model']
X_v_best   = X_val_u if best_name not in ('Logistic Regression', 'Gaussian Naive Bayes') else X_val_s

y_pred_best = best_model.predict(X_v_best)
print(f'=== Best Model: {best_name} ===')
print(classification_report(y_val, y_pred_best, target_names=['Spam (0)', 'Legit (1)']))
print(f'ROC-AUC: {roc_auc_score(y_val, best_model.predict_proba(X_v_best)[:, 1]):.4f}')

---
## 6. Save Best Model & Results

In [ ]:
os.makedirs('models', exist_ok=True)

# Retrain best model on full train set and save
best_model.fit(
    X_train_u if best_name not in ('Logistic Regression', 'Gaussian Naive Bayes') else X_train_s,
    y_train
)
joblib.dump(best_model, 'models/best_model.joblib')
joblib.dump(best_name,  'models/best_model_name.joblib')

# Save full results table
val_df.drop(columns=['_model', '_X_train'], errors='ignore').to_csv('models/val_results.csv', index=False)

print(f'Best model saved: {best_name}')
print('Files written:')
for f in os.listdir('models'):
    print(f'  models/{f}')

## Summary

| Stage | What was done |
|---|---|
| Stage 3 — Baselines | Logistic Regression, Decision Tree, Gaussian Naive Bayes — 5-fold CV |
| Stage 4 — Advanced | Random Forest, XGBoost, LightGBM — 5-fold CV |
| Tuning | RandomizedSearchCV (50 iter) on XGBoost and LightGBM |
| Selection | Best model chosen by Val F1-macro |
| Saved | `models/best_model.joblib`, `models/val_results.csv` |

**Next step → `evaluation.ipynb`**: final test-set evaluation, SHAP feature importance, error analysis.